# VAE imputation for SUPPORT (PyTorch) — single configuration

This notebook does the whole imputation pipeline in two parts:

1. **Prepare the data.** `data_imputation.load_support_missing()` loads SUPPORT with the gaps kept, encodes it and
   builds the real missingness mask. The result is saved next to this notebook (in `imputation/`):
   * `support_vae*.csv` – one row per patient: `split`, `time`, `event`, the encoded features and the `mask_` columns
     (1 = observed, 0 = missing).
   * `support_vae*_variables.json` – which columns belong to which variable, the variable type
     (`real`, `binary`, `categorical`) and the scaling needed to convert back to original units.
2. **Train the VAE and impute.** One VAE is trained on those files and fills in the missing values.

**Architecture** (`METHOD`), as in the missing-data VAE paper:

* `'Zero Imputation'` – the encoder sees the features with missing entries set to 0 (= training mean for real variables).
* `'Encoder Indicators'` – the encoder also gets the missingness mask (one indicator per variable).
* `'Encoder Decoder Indicators'` – as above, and the mask is also fed into the decoder.

**Likelihood**: every variable gets the likelihood that matches its type — Gaussian for real-valued variables,
Bernoulli for binary ones and a softmax/categorical for one-hot groups. Only *observed* variables count in the ELBO,
and each categorical variable counts once.

**Output**

* `data/support2_vae_imputed*.csv` – a copy of `data/support2.csv` with the same columns and rows, where the
  missing values of the modelled variables are filled in by the VAE (observed values are untouched). Load it for the
  survival models with `data_imputation.load_support_imputed(...)`, which runs the team's `load_support()` on it.
* `imputation/runs/` – a copy of every run, named by method and seed, in the same layout, plus an encoded version
  that keeps the `mask_` columns, so you can still see what was imputed.
* `imputation/results_support.csv` – one row of test metrics per run, for comparing methods and seeds.

The train/valid/test split is the same as `src.data.load_support()` for the same split seed.

In [1]:
import csv
import json
import math
import os
import random
import sys
import time
from datetime import datetime
from pathlib import Path

sys.path.append("..")   # project root, so that `src` can be imported

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, kl_divergence

from imputation.data_imputation import load_support_missing, split_indices
from src.data import load_support_raw

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

Using device: cuda


## Configuration

In [2]:
# ------------------------------------------------------------------
# Main choices
# ------------------------------------------------------------------
METHOD = 'Encoder Indicators'   # 'Zero Imputation', 'Encoder Indicators' or 'Encoder Decoder Indicators'
SEED = 0                        # VAE seed: model init, batching and sampling noise

# ------------------------------------------------------------------
# Data
# ------------------------------------------------------------------
DROP_POST_BASELINE = True      # True: drop sfdm2, slos, charges, totcst (recorded after study entry)
SPLIT_SEED = 42                 # train/valid/test split; 42 = same split as src.data.load_support()

# ------------------------------------------------------------------
# Model and training
# ------------------------------------------------------------------
Z_DIM = 16                      # latent dimension
HIDDEN = (128, 64)              # encoder hidden layers (decoder uses them in reverse)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 128
MAX_EPOCHS = 500
PATIENCE = 10                   # stop after this many epochs without validation-ELBO improvement
VALID_ELBO_SAMPLES = 5          # z samples per patient when computing the validation ELBO (less noisy early stopping)

# ------------------------------------------------------------------
# Evaluation and imputation
# ------------------------------------------------------------------
NUM_IMPORTANCE_SAMPLES = 500    # samples per test patient for the marginal likelihood
IMPUTE_SAMPLES = 500            # z samples per patient used to impute
IMPUTE_IMPORTANCE_WEIGHTED = True  # True: weight samples by how well they explain the observed values (MIWAE-style)
IMPUTE_HARD = True              # True: binaries -> 0/1, categoricals -> one-hot; False: keep probabilities

# ------------------------------------------------------------------
# Files (paths are relative to this notebook, i.e. the imputation/ folder)
# ------------------------------------------------------------------
IMPUTATION_DIR = Path('../imputation')
DATA_DIR = Path('../data')

# non-default data settings get a suffix, so different exports never overwrite each other
suffix = ('_baseline' if DROP_POST_BASELINE else '') + ('' if SPLIT_SEED == 42 else f'_split{SPLIT_SEED}')
DATA_FILE = IMPUTATION_DIR / f'support_vae{suffix}.csv'                  # encoded data + mask (part 1)
VARIABLES_FILE = IMPUTATION_DIR / f'support_vae{suffix}_variables.json'  # variable table (part 1)
IMPUTED_FILE = DATA_DIR / f'support2_vae_imputed{suffix}.csv'           # final imputed data, original units
RUNS_DIR = IMPUTATION_DIR / 'runs'                                       # per-run copies (method + seed in the name)
RESULTS_FILE = IMPUTATION_DIR / 'results_support.csv'                    # set to None to not save metrics
SAVE_MODEL_PATH = None          # e.g. 'vae_support.pt'

# ------------------------------------------------------------------
METHOD_OPTIONS = ('Zero Imputation', 'Encoder Indicators', 'Encoder Decoder Indicators')
assert METHOD in METHOD_OPTIONS, METHOD
print(f'SUPPORT{suffix} | method: {METHOD} | seed: {SEED} | z_dim: {Z_DIM} | hidden: {HIDDEN}')
print('data:', DATA_FILE, '|', VARIABLES_FILE)
print('output:', IMPUTED_FILE)

SUPPORT_baseline | method: Encoder Indicators | seed: 0 | z_dim: 16 | hidden: (128, 64)
data: ../imputation/support_vae_baseline.csv | ../imputation/support_vae_baseline_variables.json
output: ../data/support2_vae_imputed_baseline.csv


# Part 1 — Prepare the data

Loads SUPPORT with the missing values kept, then saves the encoded data, the mask and the variable table.

In [3]:
data = load_support_missing(drop_post_baseline=DROP_POST_BASELINE, seed=SPLIT_SEED)

print(data.name)
print(data.X_tr.shape, data.M_tr.shape)   # (6373, 50) (6373, 50) with the default settings
print('observed fraction:', data.M_tr.mean().round(3))

SUPPORT (baseline-only)
(6373, 50) (6373, 50)
observed fraction: 0.897


In [4]:
frames = []
for split, X, M, t, e in [('train', data.X_tr, data.M_tr, data.t_tr, data.e_tr),
                          ('valid', data.X_va, data.M_va, data.t_va, data.e_va),
                          ('test',  data.X_te, data.M_te, data.t_te, data.e_te)]:
  frames.append(pd.concat([pd.DataFrame({'split': split, 'time': t, 'event': e}),
                           pd.DataFrame(X, columns=data.features),
                           pd.DataFrame(M, columns=['mask_' + c for c in data.features])], axis=1))

IMPUTATION_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(frames, ignore_index=True).to_csv(DATA_FILE, index=False)

with open(VARIABLES_FILE, 'w') as f:
  json.dump([v.to_dict() for v in data.variables], f, indent=1)

print('Saved', DATA_FILE)
print('Saved', VARIABLES_FILE)

Saved ../imputation/support_vae_baseline.csv
Saved ../imputation/support_vae_baseline_variables.json


# Part 2 — Train the VAE and impute

## Load the prepared data

The features are encoded (real variables standardized with training statistics, binaries 0/1, categoricals one-hot)
and missing entries are 0. The variable table tells the model which columns belong together.

In [5]:
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)


df = pd.read_csv(DATA_FILE)
with open(VARIABLES_FILE) as f:
  VARIABLES = json.load(f)

FEATURES = [c for c in df.columns if c not in ('split', 'time', 'event') and not c.startswith('mask_')]
MASK_COLS = ['mask_' + c for c in FEATURES]
D = len(FEATURES)
N_VARS = len(VARIABLES)

# --- consistency checks between the CSV and the variable table ---
assert all(c in df.columns for c in MASK_COLS), 'mask columns missing'
assert VARIABLES[-1]['stop'] == D, 'variable table does not match the number of feature columns'
for v in VARIABLES:
  block = df[MASK_COLS[v['start']:v['stop']]].to_numpy()
  assert (block == block[:, :1]).all(), f"mask not constant within variable {v['name']}"

# --- index tensors used by the likelihood ---
REAL_VARS = [v for v in VARIABLES if v['kind'] == 'real']
BIN_VARS = [v for v in VARIABLES if v['kind'] == 'binary']
CAT_VARS = [v for v in VARIABLES if v['kind'] == 'categorical']
VAR_POS = {v['name']: i for i, v in enumerate(VARIABLES)}            # variable -> position in the per-variable mask

REAL_COLS = torch.tensor([v['start'] for v in REAL_VARS], device=DEVICE)
BIN_COLS = torch.tensor([v['start'] for v in BIN_VARS], device=DEVICE)
REAL_VPOS = torch.tensor([VAR_POS[v['name']] for v in REAL_VARS], device=DEVICE)
BIN_VPOS = torch.tensor([VAR_POS[v['name']] for v in BIN_VARS], device=DEVICE)
CAT_SLICES = [(v['start'], v['stop'], VAR_POS[v['name']]) for v in CAT_VARS]
VAR_FIRST_COL = torch.tensor([v['start'] for v in VARIABLES], device=DEVICE)  # column holding each variable's mask


def split_tensors(name):
  part = df[df['split'] == name]
  x = torch.tensor(part[FEATURES].to_numpy(np.float32), device=DEVICE)
  m = torch.tensor(part[MASK_COLS].to_numpy(np.float32), device=DEVICE)
  return x * m, m   # make sure missing entries are exactly 0


X_tr, M_tr = split_tensors('train')
X_va, M_va = split_tensors('valid')
X_te, M_te = split_tensors('test')

assert FEATURES == data.features, 'saved file does not match the prepared data'

print(f'{D} encoded columns, {N_VARS} variables '
      f'({len(REAL_VARS)} real, {len(BIN_VARS)} binary, {len(CAT_VARS)} categorical)')
for name, m in [('train', M_tr), ('valid', M_va), ('test', M_te)]:
  vm = m[:, VAR_FIRST_COL]
  print(f'{name:5s}: {m.shape[0]:5d} patients | observed variables {vm.mean().item():.3f}')

50 encoded columns, 31 variables (23 real, 3 binary, 5 categorical)
train:  6373 patients | observed variables 0.866
valid:   911 patients | observed variables 0.863
test :  1821 patients | observed variables 0.866


In [6]:
# Missingness per variable (training split)
miss = 1 - M_tr[:, VAR_FIRST_COL].mean(0).cpu().numpy()
print(pd.Series(miss, index=[v['name'] for v in VARIABLES])
        .sort_values(ascending=False).round(3).to_string())

adlp        0.618
urine       0.535
glucose     0.495
bun         0.479
alb         0.366
income      0.329
adls        0.314
bili        0.279
pafi        0.253
ph          0.249
edu         0.179
wblc        0.022
avtisst     0.010
crea        0.007
race        0.004
scoma       0.000
ca          0.000
meanbp      0.000
diabetes    0.000
hday        0.000
sex         0.000
num.co      0.000
dzclass     0.000
age         0.000
dzgroup     0.000
dementia    0.000
temp        0.000
resp        0.000
hrt         0.000
sod         0.000
adlsc       0.000


## Model

Fully connected encoder and decoder. The decoder outputs one number per encoded column: the mean for real variables,
a logit for binary variables and one logit per category for categorical variables. Each real variable also has a
learned standard deviation.

In [7]:
def mlp(sizes):
  layers = []
  for a, b in zip(sizes[:-1], sizes[1:]):
    layers += [nn.Linear(a, b), nn.ReLU()]
  return nn.Sequential(*layers)


class Encoder(nn.Module):
  def __init__(self, in_dim):
    super().__init__()
    self.body = mlp((in_dim,) + HIDDEN)
    self.mu = nn.Linear(HIDDEN[-1], Z_DIM)
    self.sigma = nn.Linear(HIDDEN[-1], Z_DIM)

  def forward(self, x):
    h = self.body(x)
    return self.mu(h), F.softplus(self.sigma(h)) + 1e-3


class Decoder(nn.Module):
  def __init__(self, b_dim=0):
    super().__init__()
    hidden = HIDDEN[::-1]
    self.first = nn.Sequential(nn.Linear(Z_DIM, hidden[0]), nn.ReLU())
    # the mask (if used) is concatenated after the first decoder layer
    self.rest = mlp((hidden[0] + b_dim,) + hidden[1:]) if len(hidden) > 1 else nn.Identity()
    self.out = nn.Linear(hidden[-1] if len(hidden) > 1 else hidden[0] + b_dim, D)
    self.log_sigma = nn.Parameter(torch.zeros(len(REAL_VARS)))  # one std per real variable

  def forward(self, z, b=None):
    h = self.first(z)
    if b is not None:
      h = torch.cat([h, b], dim=-1)
    return self.out(self.rest(h))


class VAE(nn.Module):
  def __init__(self, enc_in, dec_b):
    super().__init__()
    self.encoder = Encoder(enc_in)
    self.decoder = Decoder(dec_b)

  def real_sigma(self):
    return self.decoder.log_sigma.clamp(-4.0, 2.0).exp()


def var_mask(m):
  # one indicator per variable, from the column mask
  return m[:, VAR_FIRST_COL]


def get_inputs(x, m, method=None):
  # encoder input and (optional) decoder mask for METHOD
  method = method or METHOD
  x0 = x * m
  if method == 'Zero Imputation':
    return x0, None
  vm = var_mask(m)
  inputs = torch.cat([x0, vm], dim=-1)
  if method == 'Encoder Indicators':
    return inputs, None
  if method == 'Encoder Decoder Indicators':
    return inputs, vm
  raise ValueError(method)


ENC_IN = D + (0 if METHOD == 'Zero Imputation' else N_VARS)
DEC_B = N_VARS if METHOD == 'Encoder Decoder Indicators' else 0

## Likelihood and loss

`log_p_x_given_z` returns, for every patient, the log-likelihood of the **observed** variables only. Real variables use
a Gaussian, binaries a Bernoulli, categoricals a softmax over their one-hot group.

In [8]:
def log_p_x_given_z(model, out, x, m):
  # out: [S, B, D] or [B, D]; x, m: [B, D]. Returns the observed-data log-likelihood.
  vm = m[..., VAR_FIRST_COL]
  total = 0.0

  if len(REAL_VARS):
    mu = out[..., REAL_COLS]
    ll = Normal(mu, model.real_sigma()).log_prob(x[..., REAL_COLS])
    total = total + (ll * vm[..., REAL_VPOS]).sum(-1)

  if len(BIN_VARS):
    logits = out[..., BIN_COLS]
    target = x[..., BIN_COLS].expand_as(logits)   # works for any number of samples
    ll = -F.binary_cross_entropy_with_logits(logits, target, reduction='none')
    total = total + (ll * vm[..., BIN_VPOS]).sum(-1)

  for start, stop, pos in CAT_SLICES:
    ll = (F.log_softmax(out[..., start:stop], dim=-1) * x[..., start:stop]).sum(-1)
    total = total + ll * vm[..., pos]

  return total


def kl_to_prior(q_z):
  p_z = Normal(torch.zeros_like(q_z.loc), torch.ones_like(q_z.scale))
  return kl_divergence(q_z, p_z).sum(-1)


def elbo_terms(model, x, m, n_samples=1):
  # Returns per-patient (log p(x_o|z) averaged over samples, KL).
  inputs, dec_b = get_inputs(x, m)
  mu, sigma = model.encoder(inputs)
  q_z = Normal(mu, sigma)
  z = q_z.rsample((n_samples,))                          # [S, B, Z]
  b = None if dec_b is None else dec_b.expand(n_samples, *dec_b.shape)
  out = model.decoder(z, b)                              # [S, B, D]
  log_px = log_p_x_given_z(model, out, x, m).mean(0)     # [B]
  return log_px, kl_to_prior(q_z)

## Training and evaluation functions

In [9]:
def batches(n, batch_size, shuffle):
  order = torch.randperm(n, device=DEVICE) if shuffle else torch.arange(n, device=DEVICE)
  for start in range(0, n, batch_size):
    yield order[start:start + batch_size]


def train_epoch(model, optimizer):
  model.train()
  total, count = 0.0, 0
  for idx in batches(X_tr.shape[0], BATCH_SIZE, shuffle=True):
    log_px, kl = elbo_terms(model, X_tr[idx], M_tr[idx])
    loss = (kl - log_px).mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.5)
    optimizer.step()
    total += -loss.item() * len(idx)
    count += len(idx)
  return total / count   # training ELBO per patient


@torch.inference_mode()
def evaluate(model, X, M, n_samples=1):
  # Mean ELBO, log p(x_o|z) and KL per patient.
  model.eval()
  log_px, kl = elbo_terms(model, X, M, n_samples)
  return {'elbo': (log_px - kl).mean().item(), 'log_prob': log_px.mean().item(),
          'kl': kl.mean().item()}


@torch.inference_mode()
def posterior_samples(model, x, m, s):
  # Draws s samples z ~ q(z|x_o) and returns decoder outputs and log importance weights.
  inputs, dec_b = get_inputs(x, m)
  mu, sigma = model.encoder(inputs)
  q_z = Normal(mu, sigma)
  z = q_z.sample((s,))                                   # [S, B, Z]
  b = None if dec_b is None else dec_b.expand(s, *dec_b.shape)
  out = model.decoder(z, b)                              # [S, B, D]
  log_px = log_p_x_given_z(model, out, x, m)             # [S, B]
  log_pz = Normal(torch.zeros_like(z), torch.ones_like(z)).log_prob(z).sum(-1)
  log_qz = q_z.log_prob(z).sum(-1)
  return out, log_px + log_pz - log_qz


@torch.inference_mode()
def evaluate_test(model, X, M, s=NUM_IMPORTANCE_SAMPLES, chunk=256):
  # Importance-sampled marginal likelihood of the observed values, per patient.
  model.eval()
  mll, n_obs = [], []
  for idx in batches(X.shape[0], chunk, shuffle=False):
    _, log_w = posterior_samples(model, X[idx], M[idx], s)
    mll.append(torch.logsumexp(log_w, dim=0) - math.log(s))
    n_obs.append(var_mask(M[idx]).sum(-1))
  mll, n_obs = torch.cat(mll), torch.cat(n_obs)
  res = evaluate(model, X, M, n_samples=10)
  res['marginal_ll'] = mll.mean().item()
  res['nats_per_observed_var'] = (-mll / n_obs).mean().item()
  return res


@torch.inference_mode()
def reconstruction_check(model, X, M):
  # How well the model reproduces the *observed* values (a sanity check, not imputation quality).
  model.eval()
  inputs, dec_b = get_inputs(X, M)
  out = model.decoder(model.encoder(inputs)[0], dec_b)   # decode the posterior mean
  vm = var_mask(M)
  res = {}
  if len(REAL_VARS):
    err = (out[:, REAL_COLS] - X[:, REAL_COLS]) ** 2
    w = vm[:, REAL_VPOS]
    res['rmse_real_observed'] = (err * w).sum().div(w.sum()).sqrt().item()  # in standard deviations
  correct, total = 0.0, 0.0
  for start, stop, pos in CAT_SLICES:
    hit = (out[:, start:stop].argmax(-1) == X[:, start:stop].argmax(-1)).float()
    correct += (hit * vm[:, pos]).sum().item()
    total += vm[:, pos].sum().item()
  if len(BIN_VARS):
    hit = ((out[:, BIN_COLS] > 0).float() == X[:, BIN_COLS]).float()
    correct += (hit * vm[:, BIN_VPOS]).sum().item()
    total += vm[:, BIN_VPOS].sum().item()
  res['acc_discrete_observed'] = correct / max(total, 1)
  return res

## Train

In [10]:
set_seed(SEED)
model = VAE(ENC_IN, DEC_B).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print(f'{sum(p.numel() for p in model.parameters()):,} parameters')

best_valid_elbo, best_state, best_epoch = None, None, None
num_epochs_since_improvement = 0
t_start = time.time()

for epoch in range(MAX_EPOCHS):
  train_elbo = train_epoch(model, optimizer)
  valid_elbo = evaluate(model, X_va, M_va, n_samples=VALID_ELBO_SAMPLES)['elbo']

  improved = best_valid_elbo is None or valid_elbo > best_valid_elbo
  if improved or epoch % 10 == 0:
    print(f'epoch {epoch:3d} | train ELBO {train_elbo:9.3f} | valid ELBO {valid_elbo:9.3f}'
          f'{" *" if improved else ""}')

  if improved:
    best_valid_elbo, best_epoch = valid_elbo, epoch
    num_epochs_since_improvement = 0
    best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
  elif num_epochs_since_improvement >= PATIENCE:
    print('Early stopping.')
    break
  else:
    num_epochs_since_improvement += 1

epochs_trained = epoch + 1
train_time = time.time() - t_start
model.load_state_dict(best_state)
print(f'Trained {epochs_trained} epochs in {train_time:.0f}s; '
      f'best epoch {best_epoch} (valid ELBO {best_valid_elbo:.3f})')

if SAVE_MODEL_PATH:
  torch.save({'state_dict': best_state, 'METHOD': METHOD, 'SEED': SEED,
              'Z_DIM': Z_DIM, 'HIDDEN': HIDDEN}, SAVE_MODEL_PATH)
  print('Saved model to', SAVE_MODEL_PATH)

36,713 parameters
epoch   0 | train ELBO   -35.121 | valid ELBO   -33.973 *
epoch   1 | train ELBO   -34.049 | valid ELBO   -33.912 *
epoch   2 | train ELBO   -34.005 | valid ELBO   -33.859 *
epoch   3 | train ELBO   -33.884 | valid ELBO   -33.645 *
epoch   4 | train ELBO   -33.598 | valid ELBO   -33.368 *
epoch   5 | train ELBO   -33.371 | valid ELBO   -33.126 *
epoch   6 | train ELBO   -33.132 | valid ELBO   -32.832 *
epoch   7 | train ELBO   -32.912 | valid ELBO   -32.650 *
epoch   8 | train ELBO   -32.622 | valid ELBO   -32.405 *
epoch   9 | train ELBO   -32.388 | valid ELBO   -32.197 *
epoch  10 | train ELBO   -32.157 | valid ELBO   -32.020 *
epoch  11 | train ELBO   -32.034 | valid ELBO   -31.864 *
epoch  12 | train ELBO   -31.866 | valid ELBO   -31.706 *
epoch  13 | train ELBO   -31.785 | valid ELBO   -31.592 *
epoch  14 | train ELBO   -31.601 | valid ELBO   -31.475 *
epoch  15 | train ELBO   -31.544 | valid ELBO   -31.380 *
epoch  16 | train ELBO   -31.417 | valid ELBO   -31.27

## Test evaluation

* `marginal_ll` – importance-sampled log-likelihood of each test patient's observed values (higher is better).
  `nats_per_observed_var` divides it by the number of observed variables, so patients with many gaps are comparable.
* `rmse_real_observed` / `acc_discrete_observed` – how well the model *reconstructs observed* values. This is only a
  sanity check: the quality of the imputations themselves is judged later, through the survival models.

In [11]:
t0 = time.time()
res = evaluate_test(model, X_te, M_te)
res.update(reconstruction_check(model, X_te, M_te))
print(f'Test evaluation: {time.time() - t0:.0f}s\n')
print(f'SUPPORT | {METHOD} | seed {SEED}')
for k, v in res.items():
  print(f'  {k:24s} {v:.4f}')

Test evaluation: 0s

SUPPORT | Encoder Indicators | seed 0
  elbo                     -26.4131
  log_prob                 -18.7993
  kl                       7.6138
  marginal_ll              -25.6630
  nats_per_observed_var    0.9558
  rmse_real_observed       0.7658
  acc_discrete_observed    0.8174


## Impute

For every patient, `IMPUTE_SAMPLES` latent samples are drawn from the encoder and decoded into expected values
(real: mean; binary: probability; categorical: class probabilities). With `IMPUTE_IMPORTANCE_WEIGHTED = True` the
samples are weighted by how well they explain that patient's observed values (self-normalised importance weights, as
in MIWAE); otherwise they are averaged equally. Observed values are always kept exactly as they were.

In [12]:
def expected_values(out):
  # Decoder outputs -> expected value of every encoded column.
  ev = torch.empty_like(out)
  if len(REAL_VARS):
    ev[..., REAL_COLS] = out[..., REAL_COLS]
  if len(BIN_VARS):
    ev[..., BIN_COLS] = torch.sigmoid(out[..., BIN_COLS])
  for start, stop, _ in CAT_SLICES:
    ev[..., start:stop] = torch.softmax(out[..., start:stop], dim=-1)
  return ev


def harden(pred):
  pred = pred.clone()
  if len(BIN_VARS):
    pred[:, BIN_COLS] = (pred[:, BIN_COLS] >= 0.5).float()
  for start, stop, _ in CAT_SLICES:
    pred[:, start:stop] = F.one_hot(pred[:, start:stop].argmax(-1), stop - start).float()
  return pred


@torch.inference_mode()
def impute(model, X, M, s=IMPUTE_SAMPLES, chunk=256):
  model.eval()
  filled = []
  for idx in batches(X.shape[0], chunk, shuffle=False):
    x, m = X[idx], M[idx]
    out, log_w = posterior_samples(model, x, m, s)       # [S, B, D], [S, B]
    if IMPUTE_IMPORTANCE_WEIGHTED:
      w = torch.softmax(log_w, dim=0)
    else:
      w = torch.full_like(log_w, 1.0 / s)
    pred = (w[..., None] * expected_values(out)).sum(0)  # [B, D]
    if IMPUTE_HARD:
      pred = harden(pred)
    filled.append(m * x + (1 - m) * pred)
  return torch.cat(filled)


set_seed(SEED)
t0 = time.time()
imputed = {name: impute(model, X, M) for name, X, M in
           [('train', X_tr, M_tr), ('valid', X_va, M_va), ('test', X_te, M_te)]}
print(f'Imputation: {time.time() - t0:.0f}s')

# observed values must be untouched
for (name, X, M) in [('train', X_tr, M_tr), ('valid', X_va, M_va), ('test', X_te, M_te)]:
  assert torch.equal(imputed[name] * M, X * M), name
  assert not torch.isnan(imputed[name]).any(), name
print('Observed values unchanged, no NaNs.')

Imputation: 0s
Observed values unchanged, no NaNs.


## Convert back to original units and save

* `data/support2_vae_imputed*.csv` – **same layout as `data/support2.csv`**: all 47 columns, the original row order
  and the original index, with only the gaps filled. Observed values are copied straight from `support2.csv`, so they
  are exactly the original numbers. Columns the VAE did not model (e.g. `death`, `d.time`, `totmcst`) are left as they
  are; `load_support()` drops them anyway. It is overwritten by every run, so it always holds the latest one.
* `imputation/runs/support2_vae_imputed*_<method>_seed<seed>.csv` – a copy of this run in the same layout, and
  `..._encoded.csv` – the encoded version with the `mask_` columns.

In [13]:
def to_original_units(X):
  # Encoded matrix (numpy, [n, D]) -> DataFrame with one column per original variable.
  out = {}
  for v in VARIABLES:
    block = X[:, v['start']:v['stop']]
    if v['kind'] == 'real':
      x = block[:, 0] * v['std'] + v['mean']
      out[v['name']] = np.expm1(x) if v['log'] else x
    elif v['kind'] == 'binary':
      pos = v['positive']
      neg = {'male': 'female', 1: 0}.get(pos, f'not {pos}')
      out[v['name']] = np.where(block[:, 0] >= 0.5, pos, neg)
    else:
      out[v['name']] = np.asarray(v['categories'], dtype=object)[block.argmax(1)]
  return pd.DataFrame(out)


tag = f"{suffix}_{METHOD.lower().replace(' ', '_')}_seed{SEED}"
encoded_parts, original_parts = [], []
for name in ['train', 'valid', 'test']:
  part = df[df['split'] == name].reset_index(drop=True)
  Ximp = imputed[name].cpu().numpy()
  meta = part[['split', 'time', 'event']]
  encoded_parts.append(pd.concat([meta, pd.DataFrame(Ximp, columns=FEATURES), part[MASK_COLS]], axis=1))
  original_parts.append(pd.concat([meta, to_original_units(Ximp)], axis=1))

original = pd.concat(original_parts, ignore_index=True)   # rows ordered train, valid, test

# --- put the imputed values back into the support2.csv layout ---
raw = load_support_raw()
order = np.concatenate(split_indices(raw['death'].to_numpy(int), SPLIT_SEED))  # raw row of each row of `original`
assert np.array_equal(raw['d.time'].to_numpy(float)[order], original['time'].to_numpy(float))
filled = original.drop(columns=['split', 'time', 'event'])
filled.index = raw.index[order]
filled = filled.loc[raw.index]                            # back to the original row order

support2_imputed = raw.copy()
for c in filled.columns:
  support2_imputed[c] = raw[c].where(raw[c].notna(), filled[c])   # keep observed values, fill only the gaps
assert support2_imputed[filled.columns].notna().all().all()
assert list(support2_imputed.columns) == list(raw.columns)

RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
original_path = RUNS_DIR / f'support2_vae_imputed{tag}.csv'
encoded_path = RUNS_DIR / f'support2_vae_imputed{tag}_encoded.csv'
support2_imputed.to_csv(IMPUTED_FILE)
support2_imputed.to_csv(original_path)
pd.concat(encoded_parts, ignore_index=True).to_csv(encoded_path, index=False)
print('Saved', IMPUTED_FILE, support2_imputed.shape)
print('Saved', original_path)
print('Saved', encoded_path)


Saved ../data/support2_vae_imputed_baseline.csv (9105, 47)
Saved ../imputation/runs/support2_vae_imputed_baseline_encoder_indicators_seed0.csv
Saved ../imputation/runs/support2_vae_imputed_baseline_encoder_indicators_seed0_encoded.csv


In [14]:
# Look at a few test patients: imputed values are marked with *
part = df[df['split'] == 'test'].reset_index(drop=True)
orig = to_original_units(imputed['test'].cpu().numpy())
vmask = var_mask(M_te).cpu().numpy().astype(bool)
show = [v['name'] for v in VARIABLES][:14]
rows = []
for i in range(5):
  row = {}
  for j, v in enumerate(VARIABLES):
    if v['name'] in show:
      val = orig.loc[i, v['name']]
      val = f'{val:.2f}' if isinstance(val, (float, np.floating)) else str(val)
      row[v['name']] = val if vmask[i, j] else val + ' *'
  rows.append(row)
print(pd.DataFrame(rows).to_string())

     age     sex            dzgroup             dzclass num.co      edu        income   scoma avtisst   race   hday diabetes dementia   ca
0  78.79  female  ARF/MOSF w/Sepsis            ARF/MOSF   2.00  12.19 *  under $11k *   41.00   14.50  black  15.00        1        0  yes
1  32.97    male               Coma                Coma   0.00    14.00      $11-$25k  100.00   36.67  white   4.00        0        0   no
2  80.46  female  ARF/MOSF w/Sepsis            ARF/MOSF   3.00     8.00    under $11k    0.00   24.67  white   1.00        0        1   no
3  39.64    male          Cirrhosis  COPD/CHF/Cirrhosis   2.00    13.00    $25-$50k *    0.00   13.75  white   1.00        0        0   no
4  48.00    male                CHF  COPD/CHF/Cirrhosis   1.00  11.86 *  under $11k *    0.00   33.50  white   1.00        0        0   no


## Save metrics

In [15]:
METRIC_NAMES = ['elbo', 'log_prob', 'kl', 'marginal_ll', 'nats_per_observed_var',
                'rmse_real_observed', 'acc_discrete_observed']

if RESULTS_FILE:
  row = {'timestamp': datetime.now().isoformat(timespec='seconds'),
         'data': f'support{suffix}', 'method': METHOD, 'seed': SEED, 'z_dim': Z_DIM, 'hidden': '-'.join(map(str, HIDDEN)),
         'max_epochs': MAX_EPOCHS, 'patience': PATIENCE,
         'epochs_trained': epochs_trained, 'best_epoch': best_epoch,
         'train_time_s': round(train_time, 1), 'imputed_file': str(original_path)}
  row.update({name: res.get(name, float('nan')) for name in METRIC_NAMES})
  new_file = not os.path.exists(RESULTS_FILE)
  with open(RESULTS_FILE, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(row))
    if new_file:
      writer.writeheader()
    writer.writerow(row)
  print('Appended results to', RESULTS_FILE)

Appended results to ../imputation/results_support.csv
